# Homework 3: Machine Learning for Classification

ML Zoomcamp 2026 · Dataset: `course_lead_scoring_2026.csv` · Target: `converted`

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Preparación de datos

Categóricas faltantes → `'NA'`, numéricas faltantes → `0.0`.

In [3]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
categorical = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
numerical = [c for c in df.columns if c not in categorical and c != 'converted']

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)
print('Categóricas:', categorical)
print('Numéricas:', numerical)
df.isnull().sum().sum()

Categóricas: ['lead_source', 'industry', 'employment_status', 'location']
Numéricas: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


np.int64(0)

## Pregunta 1: moda de `industry`

In [5]:
df['industry'].value_counts().head()

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64

In [6]:
q1 = df['industry'].mode()[0]
q1

'technology'

## Pregunta 2: mayor correlación

In [7]:
corr = df[numerical].corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [8]:
pairs = [('interaction_count', 'lead_score'),
         ('number_of_courses_viewed', 'lead_score'),
         ('number_of_courses_viewed', 'interaction_count'),
         ('annual_income', 'interaction_count')]
for a, b in pairs:
    print(f'{a} - {b}: {corr.loc[a, b]:.4f}')
q2 = max(pairs, key=lambda p: abs(corr.loc[p]))
q2

interaction_count - lead_score: 0.9157
number_of_courses_viewed - lead_score: 0.7572
number_of_courses_viewed - interaction_count: 0.7216
annual_income - interaction_count: 0.1228


('interaction_count', 'lead_score')

## División de datos (60/20/20, `random_state=42`)

In [9]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

df_train = df_train.drop(columns='converted').reset_index(drop=True)
df_val = df_val.drop(columns='converted').reset_index(drop=True)
df_test = df_test.drop(columns='converted').reset_index(drop=True)
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Pregunta 3: mutual information (solo train)

In [10]:
mi = {c: round(mutual_info_score(df_train[c], y_train), 2) for c in categorical}
mi_raw = {c: mutual_info_score(df_train[c], y_train) for c in categorical}
pd.DataFrame({'mi_rounded': mi, 'mi_raw': mi_raw}).sort_values('mi_raw', ascending=False)

,mi_rounded,mi_raw
lead_source,0.03,0.034346
employment_status,0.02,0.018966
industry,0.00,0.001995
location,0.00,0.001043


## Pregunta 4: regresión logística con one-hot encoding

In [11]:
def train_eval(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

features = categorical + numerical
base_acc = train_eval(features)
print('Accuracy:', base_acc)
round(base_acc, 2)

Accuracy: 0.645


0.65

## Pregunta 5: eliminación de características

Diferencia = accuracy original − accuracy sin la característica (puede ser negativa).

In [12]:
diffs = {}
for f in features:
    acc = train_eval([x for x in features if x != f])
    diffs[f] = base_acc - acc
res = pd.Series(diffs).sort_values()
res

annual_income              -0.079
industry                    0.000
location                    0.000
employment_status           0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

In [13]:
candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
res[candidates].sort_values()

number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

## Pregunta 6: regularización (parámetro C)

In [14]:
scores = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    scores[C] = round(train_eval(features, C=C), 3)
    print(C, scores[C])
best = max(scores.values())
q6 = min(C for C, s in scores.items() if s == best)
q6

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


0.001

## Respuestas

| # | Pregunta | Respuesta |
|---|---|---|
| 1 | Moda de `industry` | `technology` |
| 2 | Mayor correlación | `interaction_count` y `lead_score` (0.916) |
| 3 | Mayor MI | `lead_source` (0.03) |
| 4 | Accuracy | 0.65 |
| 5 | Menor diferencia | `number_of_courses_viewed` (0.002) |
| 6 | Mejor C | 0.001 |